In [1]:
# orders # /public/trendytech/retail_db/orders
# ========
# order_id,order_date,order_order_customer_id,order_status
# 1,2013-07-25 00:00:00.0,11599,CLOSED
# 2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT
# 3,2013-07-25 00:00:00.0,12111,COMPLETE

# customers # /public/trendytech/retail_db/customers
# ==========
# customer_id,customer_fname,customer_lname,customer_email,customer_password,customer_street,customer_city,customer_state,customer_zipcode
# 1,Richard,Hernandez,XXXXXXXXX,XXXXXXXXX,6303 Heather Plaza,Brownsville,TX,78521
# 2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126
# 3,Ann,Smith,XXXXXXXXX,XXXXXXXXX,3422 Blue Pioneer Bend,Caguas,PR,00725

# order_items # /public/trendytech/retail_db/order_items
# ==========
# order_item_id,order_id,order_item_product_id,order_item_quantity,order_item_subtotal,order_item_product_price
# 1,1,957,1,299.98,299.98
# 2,2,1073,1,199.99,199.99
# 3,2,502,5,250.0,50.0
# 4,2,403,1,129.99,129.99

In [2]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession. \
builder. \
appName("week4-assignments"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [3]:
orders = spark.sparkContext.textFile("/public/trendytech/retail_db/orders/part-00000")

In [4]:
customers = spark.sparkContext.textFile("/public/trendytech/retail_db/customers/part-00000")

In [5]:
order_items = spark.sparkContext.textFile("/public/trendytech/retail_db/order_items/part-00000")

In [6]:
## 1. we need to find top 10 customers who have spent the most amount(premium customers)

In [7]:
orders1 = orders.map(lambda x: (x.split(",")[0],x.split(",")[2]))  #(order_id,order_customer_id)

In [8]:
#orders1.take(3)

In [9]:
order_items1 = order_items.map(lambda x: (x.split(",")[1],float(x.split(",")[4]))) #(order_id,order_item_subtotal)

In [10]:
#order_items1.take(3)

In [11]:
order_cust_item = orders1.join(order_items1)  #### (order_id ,(order_customer_id,order_item_subtotal))

In [59]:
order_cust_item.take(3)

[('34566', ('3066', 250.0)),
 ('34566', ('3066', 179.97)),
 ('34577', ('7733', 299.98))]

In [13]:
cust_total = order_cust_item.map(lambda x: x[1]) ### (order_customer_id,order_item_subtotal)

In [60]:
cust_total.take(3)

[('3066', 250.0), ('3066', 179.97), ('7733', 299.98)]

In [15]:
cust_total_sum = cust_total.reduceByKey(lambda x,y : x+y).sortBy(lambda x:x[1],False)

In [61]:
cust_total_sum.take(10)

[('791', 10524.169999999998),
 ('9371', 9299.029999999999),
 ('8766', 9296.14),
 ('1657', 9223.71),
 ('2641', 9130.92),
 ('1288', 9019.11),
 ('3710', 9019.099999999999),
 ('4249', 8918.85),
 ('5654', 8904.95),
 ('5624', 8761.98)]

In [17]:
#2. top 10 product id's with most quantities sold

In [18]:
order_items2 = order_items.map(lambda x: (x.split(",")[2],float(x.split(",")[3]))) #(order_item_product_id,order_item_quantity)

In [19]:
#order_items2.take(3)

In [20]:
item_qty = order_items2.reduceByKey(lambda x,y : x+y)

In [21]:
#item_qty.take(3)

In [22]:
top_10 = item_qty.sortBy(lambda x:x[1],False)

In [23]:
top_10.take(5)

[('365', 73698.0),
 ('502', 62956.0),
 ('1014', 57803.0),
 ('191', 36680.0),
 ('627', 31735.0)]

In [24]:
#3. how many customers are from Caguas city

In [25]:
customers1 = customers.map(lambda x: (x.split(",")[6],x.split(",")[0]))  #(customer_city,customer_id)

In [26]:
#customers1.take(4)

In [27]:
caguas_cust = customers1.filter(lambda x:x[0]=='Caguas')

In [28]:
caguas_cust.count()

4584

In [29]:
#4. Top 3 states with maximum customers

In [30]:
customers2 = customers.map(lambda x: (x.split(",")[7],1))  #(customer_state,1)

In [31]:
#customers2.take(5)

In [32]:
state_cust_count = customers2.reduceByKey(lambda x,y : x+y)

In [33]:
#state_cust_count.take(4)

In [34]:
top3state = state_cust_count.sortBy(lambda x:x[1],False)

In [35]:
top3state.take(3)

[('PR', 4771), ('CA', 2012), ('NY', 775)]

In [36]:
#5.how many customers have spent more than $1000 in total

In [37]:
cust_gt_1000 = cust_total_sum.filter(lambda x: x[1] > 1000)

In [38]:
cust_gt_1000.count()

11148

In [39]:
cust_gt_1000.take(5)

[('791', 10524.169999999998),
 ('9371', 9299.029999999999),
 ('8766', 9296.14),
 ('1657', 9223.71),
 ('2641', 9130.92)]

In [40]:
#6. which state has most number of orders in CLOSED status

In [41]:
cust_status = orders.map(lambda x: (x.split(",")[2],x.split(",")[3]))  #(order_customer_id,order_status)

In [42]:
cust_status.take(5)

[('11599', 'CLOSED'),
 ('256', 'PENDING_PAYMENT'),
 ('12111', 'COMPLETE'),
 ('8827', 'CLOSED'),
 ('11318', 'COMPLETE')]

In [43]:
customers3 = customers.map(lambda x: (x.split(",")[0],x.split(",")[7]))  #(customer_id,customer_state)

In [44]:
customers3.take(5)

[('1', 'TX'), ('2', 'CO'), ('3', 'PR'), ('4', 'CA'), ('5', 'PR')]

In [45]:
state_status = cust_status.join(customers3)

In [46]:
state_status.take(5)

[('2248', ('PROCESSING', 'PR')),
 ('2248', ('ON_HOLD', 'PR')),
 ('2248', ('CLOSED', 'PR')),
 ('2248', ('COMPLETE', 'PR')),
 ('7733', ('CANCELED', 'CA'))]

In [47]:
state_status1 = state_status.filter(lambda x: x[1][0]=='CLOSED').map(lambda x: (x[1][1],1))

In [48]:
state_status1.take(5)

[('PR', 1), ('CA', 1), ('MI', 1), ('MI', 1), ('CA', 1)]

In [49]:
most_closed_state = state_status1.reduceByKey(lambda x,y:x+y).sortBy(lambda x:x[1],False)

In [50]:
most_closed_state.take(5)

[('PR', 2891), ('CA', 1232), ('NY', 450), ('TX', 403), ('IL', 313)]

In [51]:
# 7 how many customers are active (active customers are the one's who placed at least one order)

In [52]:
cust_order1 = orders.map(lambda x: x.split(",")[2])  #(order_id,order_customer_id)

In [57]:
cust_order1.take(2)

['11599', '256']

In [56]:
cust_order1.distinct().count()

12405

In [58]:
# 8. What is the revenue generated by each state in sorted order

In [62]:
cust_total_sum.take(3)

[('791', 10524.169999999998), ('9371', 9299.029999999999), ('8766', 9296.14)]

In [63]:
customers.take(3)

['1,Richard,Hernandez,XXXXXXXXX,XXXXXXXXX,6303 Heather Plaza,Brownsville,TX,78521',
 '2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126',
 '3,Ann,Smith,XXXXXXXXX,XXXXXXXXX,3422 Blue Pioneer Bend,Caguas,PR,00725']

In [64]:
state_cust_id = customers.map(lambda x: (x.split(",")[0],x.split(",")[7]))

In [65]:
state_cust_id.take(3)

[('1', 'TX'), ('2', 'CO'), ('3', 'PR')]

In [66]:
state_tot = state_cust_id.join(cust_total_sum)

In [67]:
state_tot.take(3)

[('4', ('CA', 1719.63)),
 ('16', ('PR', 2349.7700000000004)),
 ('20', ('NJ', 1589.77))]

In [68]:
state_rev = state_tot.map(lambda x: (x[1][0],x[1][1]))

In [69]:
state_rev.take(3)

[('CA', 1719.63), ('PR', 2349.7700000000004), ('NJ', 1589.77)]

In [70]:
state_rev_sum = state_rev.reduceByKey(lambda x,y : x+y).sortBy(lambda x:x[1],False)

In [71]:
state_rev_sum.take(5)

[('PR', 13208867.690000001),
 ('CA', 5542722.999999999),
 ('NY', 2152706.74),
 ('TX', 1731407.49),
 ('IL', 1457225.83)]